# LLM Reasoning Comparison


## SECTION 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
path='/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
print(os.listdir(path))


## SECTION 2 — Install Dependencies


In [ ]:
!pip install -q transformers pandas numpy matplotlib seaborn torch


## SECTION 3 — Load Dataset


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
required = ['X_train.npy','X_test.npy','y_train.npy','y_test.npy']
print('Data path:', DATA_PATH)
print('Files:', os.listdir(DATA_PATH))
missing = [f for f in required if not os.path.exists(os.path.join(DATA_PATH,f))]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

X_test=np.load(os.path.join(DATA_PATH,'X_test.npy'))
y_test=np.load(os.path.join(DATA_PATH,'y_test.npy')).reshape(-1)
if X_test.ndim==3 and X_test.shape[-1]==1: X_test=X_test[...,0]
feature_names=[f'feature_{i}' for i in range(X_test.shape[1])]
print('number of samples:',X_test.shape[0])
print('feature names (first 20):',feature_names[:20])
print('attack classes:',sorted(np.unique(y_test).tolist()))



## SECTION 4 — Select Sample


In [ ]:
sample_index=int(input('Enter traffic sample index: '))
sample_index=max(0,min(sample_index,X_test.shape[0]-1))
x=X_test[sample_index]; y=int(y_test[sample_index])
print('Selected sample:',sample_index,'class=',y)
display(pd.DataFrame({'feature':feature_names[:20],'value':x[:20]}))



## SECTION 5 — Simulate IDS Context


In [ ]:
attack_map={0:'BENIGN',1:'DOS',2:'PORTSCAN',3:'BRUTEFORCE',4:'DDOS'}
attack_type=attack_map.get(y,f'ATTACK_{y}')
confidence=float(np.clip(0.6+0.25*(y>0)+np.random.normal(0,0.08),0,1))
risk_score=float(np.clip(0.55*confidence+0.35*(y>0)+np.random.normal(0,0.06),0,1))
idx=np.argsort(np.abs(x))[-5:][::-1]
top_features=[feature_names[i] for i in idx]
memory_neighbors=[f'memory_case_{i}' for i in np.random.choice(range(100),size=3,replace=False)]
graph_neighbors=np.random.choice(['DOS','DDOS','PORTSCAN','INFILTRATION','BOTNET'],size=3,replace=False).tolist()
ctx={'attack_type':attack_type,'confidence':confidence,'risk_score':risk_score,'top_features':top_features,'memory_neighbors':memory_neighbors,'graph_neighbors':graph_neighbors}
ctx



## SECTION 6 — LLM Reasoning Methods


In [ ]:
import time
from transformers import pipeline

def template(c):
    return f"Template: {c['attack_type']} confidence={c['confidence']:.2f} risk={c['risk_score']:.2f}; features={c['top_features']}; memory={c['memory_neighbors']}; graph={c['graph_neighbors']}."
def cot(c):
    return f"Step1 classify {c['attack_type']}. Step2 inspect features {c['top_features']}. Step3 compare memory {c['memory_neighbors']}. Step4 inspect graph {c['graph_neighbors']}. Step5 action by risk {c['risk_score']:.2f}."
def rag(c):
    kb={'DOS':'DoS degrades availability by floods.','DDOS':'DDoS uses distributed sources.','PORTSCAN':'Portscan indicates reconnaissance.','BRUTEFORCE':'Bruteforce indicates credential abuse.','BENIGN':'Benign traffic has low malicious overlap.'}
    return f"RAG: {kb.get(c['attack_type'],'Limited KB match.')} Context confidence={c['confidence']:.2f}, risk={c['risk_score']:.2f}, features={c['top_features']}."

gen=None
try:
    gen=pipeline('text2text-generation',model='google/flan-t5-base')
except Exception as e:
    print('Flan-T5 unavailable, using fallback:',e)

def flan(c):
    prompt=f"Explain IDS alert attack={c['attack_type']} confidence={c['confidence']:.2f} risk={c['risk_score']:.2f} features={c['top_features']} memory={c['memory_neighbors']} graph={c['graph_neighbors']}"
    if gen is None: return 'Flan fallback: summary from given context.'
    return gen(prompt,max_length=140,do_sample=False)[0]['generated_text']

outs={}; times={}
for name,fn in [('Template',template),('ChainOfThought',cot),('RAG',rag),('FlanT5',flan)]:
    t0=time.perf_counter(); outs[name]=fn(ctx); times[name]=time.perf_counter()-t0
    print(f"\n===== {name} =====")
    print(outs[name])



## SECTION 7 — Comparison


In [ ]:
def clarity(t):
    s=t.lower(); kws=['risk','confidence','feature','memory','graph','attack','action']
    return 0.7*sum(k in s for k in kws)/len(kws)+0.3*min(len(t.split())/60,1)
rows=[]
for m,t in outs.items():
    rows.append({'Method':m,'ExplanationLength':len(t.split()),'RuntimeSec':times[m],'ReasoningClarity':clarity(t)})
cmp=pd.DataFrame(rows).sort_values(['ReasoningClarity','RuntimeSec'],ascending=[False,True]).reset_index(drop=True)
cmp



## SECTION 8 — Visualization


In [ ]:
cmp.plot(x='Method',y='RuntimeSec',kind='bar',figsize=(8,4),title='Runtime Comparison')
plt.tight_layout(); plt.show()



## SECTION 9 — Print Best Method


In [ ]:
cmp['CompositeScore']=cmp['ReasoningClarity']-0.2*(cmp['RuntimeSec']/max(cmp['RuntimeSec'].max(),1e-8))
best=cmp.sort_values('CompositeScore',ascending=False).iloc[0]
print('Best reasoning strategy:',best['Method'])
print(best.to_string())
cmp

